### Leer todos los datos que son requeridos

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
movies_df = spark.read.table("movie_silver.movies") \
    .filter(f"file_date = '{v_file_date}'")

In [0]:
country_df = spark.read.table("movie_silver.countries")

In [0]:
production_country_df = spark.read.table("movie_silver.productions_countries") \
    .filter(f"file_date = '{v_file_date}'")

In [0]:
production_company_df = spark.read.table("movie_silver.productions_companies") \
    .filter(f"file_date = '{v_file_date}'")

In [0]:
movie_company_df = spark.read.table("movie_silver.movies_companies") \
    .filter(f"file_date = '{v_file_date}'")

#### Join "country" y "production_country"

In [0]:
country_prod_coun_df = country_df.join(production_country_df,
                                         country_df.country_id == production_country_df.country_id,
                                         "inner") \
                        .select(country_df.country_name, country_df.country_id,
                                production_country_df.movie_id)

#### Join "production_company" y "movie_company"

In [0]:
prod_com_mov_comp_df = production_company_df.join(movie_company_df,
                                   production_company_df.company_id == movie_company_df.company_id,
                                   "inner") \
                    .select(production_company_df.company_name, production_company_df.company_id,
                            movie_company_df.movie_id)

#### Join "movies_df", "country_prod_coun_df" y "prod_com_mov_comp_df"

- Filtrar las películas donde su fecha de lanzamiento sea mayor o igual a 2010

In [0]:
movie_filter_df = movies_df.filter("year_release_date >= 2010")

In [0]:
results_movies_country_prod_company_df = movie_filter_df.join(country_prod_coun_df,
                                    movie_filter_df.movie_id == country_prod_coun_df.movie_id,
                                    "inner") \
                                                    .join(prod_com_mov_comp_df,
                                    movie_filter_df.movie_id == prod_com_mov_comp_df.movie_id,
                                    "inner")

- Agregar la columna "created_date"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
results_df = results_movies_country_prod_company_df \
            .select(movie_filter_df.movie_id, "country_id", "company_id", "title", "budget", "revenue", "duration_time", "release_date", "country_name", "company_name") \
            .withColumn("created_date", lit(v_file_date))

- Ordenar por la columna "title" de manera descendente

In [0]:
results_order_by_df = results_df.orderBy(results_df.title.asc())

#### Escribir datos en el DataLake en formato "Delta"

In [0]:
#overwrite_partition("movie_gold", "results_country_prod_company", "created_date", v_file_date)

In [0]:
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id AND tgt.created_date = src.created_date'
merge_delta_lake(results_order_by_df, "movie_gold", "results_country_prod_company", merge_condition, "created_date")

In [0]:
%sql
SELECT * FROM movie_gold.results_country_prod_company;

movie_id,country_id,company_id,title,budget,revenue,duration_time,release_date,country_name,company_name,created_date
301325,214,75278,#Horror,1500000,1875000.0,90,2015-11-20,United States of America,Lowland Pictures,2024-12-30
301325,214,75277,#Horror,1500000,1875000.0,90,2015-11-20,United States of America,AST Studios,2024-12-30
433715,214,85248,8 Days,2000000,2600000.0,90,2014-06-15,United States of America,After Eden Pictures,2024-12-30
343795,214,68117,90 Minutes in Heaven,5000000,4842699.0,121,2015-09-11,United States of America,Giving Films,2024-12-30
241239,128,53656,A Most Violent Year,20000000,1.200707E7,125,2014-12-30,United Arab Emirates,Old Bull Pictures,2024-12-30
241239,128,41077,A Most Violent Year,20000000,1.200707E7,125,2014-12-30,United Arab Emirates,A24,2024-12-30
241239,214,53656,A Most Violent Year,20000000,1.200707E7,125,2014-12-30,United States of America,Old Bull Pictures,2024-12-30
241239,214,41077,A Most Violent Year,20000000,1.200707E7,125,2014-12-30,United States of America,A24,2024-12-30
169917,214,40107,A Walk Among the Tombstones,28000000,5.31816E7,113,2014-09-18,United States of America,Da Vinci Media Ventures,2024-12-30
169917,214,40106,A Walk Among the Tombstones,28000000,5.31816E7,113,2014-09-18,United States of America,Free State Pictures,2024-12-30
